# Analysis of Misclassifications

In [61]:
from pathlib import Path
import json
import pickle

import pandas as pd

In [62]:
# Experiment config

dataset = "darpa2000"
scenario = "s1_dmz"

logic_file = "darpa_test"
train_mode = "scratch" # "scratch" or "pretrained"
dataset_variant = "up" # "original" or "resampled"
window_size = 10

run_id = "20260402_114027"

## Compute Misclassifications

### Load Original Flows

In [63]:
df = pd.read_csv(
    f"../data/interim/{dataset}/{scenario}/flows_labeled/all_flows_labeled.csv"
)

df = df.sort_values("start_time").reset_index(drop=True)
df['t_rel'] = df['start_time'] - df['start_time'].min()

### Load DPL Dataset

In [64]:
def load_dpl_dataset(logic_file, cache_file_name):
    dpl_dataset_dir = Path(f"../experiments/{dataset}/{scenario}/deepproblog/{logic_file}/cache/")
    cache_file_test = dpl_dataset_dir / cache_file_name

    cache = pickle.load(open(cache_file_test, "rb"))
    cache_df = pd.DataFrame(cache)
    cache_df.head()

    return cache_df

In [65]:
cache_file_name = f"{logic_file}_{dataset_variant}_w{window_size}_test.pkl"
cache_df = load_dpl_dataset(logic_file, cache_file_name)

### Map Misclassifications to Original Flows

In [66]:
errors_dir = f"../experiments/{dataset}/{scenario}/deepproblog/{logic_file}/errors"
experiment_name = f"{logic_file}_{train_mode}_{dataset_variant}_w{window_size}"

errors_file = (
    f"{errors_dir}/"
    f"{experiment_name}_{run_id}.json"
)
    
with open(errors_file) as f:
    errors = json.load(f)

In [67]:
dpl_to_orig = dict(zip(cache_df['dpl_index'], cache_df['orig_index']))

original_indices = []
mis_y_preds = []
mis_y_trues = []

phase_map = {
    "benign": 0,
    "phase1": 1,
    "phase2": 2,
    "phase3": 3,
    "phase4": 4,
    "phase5": 5,
}

for error in errors:
    dpl_index = error['index']

    original_indices.append(dpl_to_orig[dpl_index])
    y_pred = error['predicted']
    y_true = error['actual']
    mis_y_preds.append(phase_map[y_pred])
    mis_y_trues.append(phase_map[y_true])

mis_df = df.loc[original_indices].copy()
mis_df['y_pred'] = mis_y_preds
mis_df['y_true'] = mis_y_trues

In [68]:
mis_df

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,phase,t_rel,y_pred,y_true
3170,f3157,9.524417e+08,9.524417e+08,0.012747,197.218.177.69,8234,172.16.112.100,80,tcp,http,...,5,516,4,1571,-,6,0,2858.728210,3,0
3173,f3160,9.524417e+08,9.524417e+08,0.013162,197.218.177.69,8242,172.16.112.100,80,tcp,http,...,5,516,4,2430,-,6,0,2858.782267,3,0
3174,f3161,9.524417e+08,9.524417e+08,0.011815,197.218.177.69,8243,172.16.112.100,80,tcp,http,...,5,516,4,1266,-,6,0,2858.797550,3,0
3206,f3193,9.524417e+08,9.524417e+08,0.025816,197.218.177.69,8245,172.16.112.100,80,tcp,http,...,6,564,6,3493,-,6,0,2881.681086,3,0
3207,f3194,9.524417e+08,9.524417e+08,0.013044,197.218.177.69,8246,172.16.112.100,80,tcp,http,...,5,520,4,1571,-,6,0,2881.709419,3,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41914,f41891,9.524470e+08,9.524470e+08,0.133419,172.16.116.201,49611,131.84.1.31,80,tcp,http,...,11,669,30,38457,-,6,5,8148.355546,0,5
42023,f41997,9.524471e+08,9.524471e+08,0.054359,172.16.115.5,49699,131.84.1.31,80,tcp,http,...,5,549,6,2035,-,6,5,8282.952414,0,5
42114,f42088,9.524473e+08,9.524473e+08,0.025293,172.16.115.5,49777,131.84.1.31,80,tcp,http,...,5,527,5,2472,-,6,5,8423.345602,0,5
42115,f42089,9.524473e+08,9.524473e+08,0.048319,172.16.115.5,49778,131.84.1.31,80,tcp,http,...,7,597,9,6530,-,6,5,8423.379288,0,5


## Misclassification Analysis

### Helper functions

In [69]:
def false_positives(df, phase):
    return df[(df["y_true"] != phase) & (df["y_pred"] == phase)]

def false_negatives(df, phase):
    return df[(df["y_true"] == phase) & (df["y_pred"] != phase)]

def true_positives(df, phase):
    return df[(df["y_true"] == phase) & (df["y_pred"] == phase)]

### Per-Phase Misclassifications

#### Phase 2

In [70]:
df_fp_2 = false_positives(mis_df, 2)
df_fp_2

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,phase,t_rel,y_pred,y_true


In [71]:
df_fn_2 = false_negatives(mis_df, 2)
df_fn_2

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,phase,t_rel,y_pred,y_true


In [72]:
df_fn_2["local_orig"]
df_fn_2["local_resp"]

Series([], Name: local_resp, dtype: object)

In [73]:
df_tp_2 = true_positives(mis_df, 2)
df_tp_2

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,phase,t_rel,y_pred,y_true


#### Phase 3

In [74]:
df_fn_3 = false_negatives(mis_df, 3)
df_fn_3

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,phase,t_rel,y_pred,y_true


In [75]:
df_fp_3 = false_positives(mis_df, 3)
df_fp_3

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,phase,t_rel,y_pred,y_true
3170,f3157,9.524417e+08,9.524417e+08,0.012747,197.218.177.69,8234,172.16.112.100,80,tcp,http,...,5,516,4,1571,-,6,0,2858.728210,3,0
3173,f3160,9.524417e+08,9.524417e+08,0.013162,197.218.177.69,8242,172.16.112.100,80,tcp,http,...,5,516,4,2430,-,6,0,2858.782267,3,0
3174,f3161,9.524417e+08,9.524417e+08,0.011815,197.218.177.69,8243,172.16.112.100,80,tcp,http,...,5,516,4,1266,-,6,0,2858.797550,3,0
3206,f3193,9.524417e+08,9.524417e+08,0.025816,197.218.177.69,8245,172.16.112.100,80,tcp,http,...,6,564,6,3493,-,6,0,2881.681086,3,0
3207,f3194,9.524417e+08,9.524417e+08,0.013044,197.218.177.69,8246,172.16.112.100,80,tcp,http,...,5,520,4,1571,-,6,0,2881.709419,3,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4537,f4520,9.524432e+08,9.524432e+08,0.015604,194.7.248.153,11951,172.16.114.50,80,tcp,http,...,5,509,4,1544,-,6,0,4324.216377,3,0
4539,f4521,9.524432e+08,9.524432e+08,0.016466,194.7.248.153,12015,172.16.114.50,80,tcp,http,...,5,511,5,2902,-,6,0,4324.234217,3,0
4541,f4522,9.524432e+08,9.524432e+08,0.016429,194.7.248.153,12076,172.16.114.50,80,tcp,http,...,5,509,5,2711,-,6,0,4324.252828,3,0
4545,f4524,9.524432e+08,9.524432e+08,0.014970,194.7.248.153,12140,172.16.114.50,80,tcp,http,...,5,509,4,1239,-,6,0,4324.289972,3,0


#### Phase 4

In [76]:
df_fn_4 = false_negatives(mis_df, 4)
df_fn_4

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,phase,t_rel,y_pred,y_true
5516,f5910,9.524442e+08,9.524442e+08,0.001277,172.16.112.10,1023,202.77.162.213,1018,tcp,-,...,2,100,1,44,-,6,4,5381.81526,0,4


In [77]:
df_fp_4 = false_positives(mis_df, 4)
df_fp_4

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,phase,t_rel,y_pred,y_true
5180,f45437,9.524438e+08,9.524506e+08,6808.971401,197.182.91.233,13449,172.16.115.20,23,tcp,-,...,3171,128508,1646,75839,-,6,0,4936.829975,4,0


In [78]:
df_fp_4["dport"].value_counts()

dport
23    1
Name: count, dtype: int64

#### Phase 5

In [79]:
df_fn_5 = false_negatives(mis_df, 5)
df_fn_5

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,phase,t_rel,y_pred,y_true
7593,f7570,9.524465e+08,9.524465e+08,0.000000,131.84.1.31,31938,61.56.80.155,4940,tcp,-,...,1,40,0,0,-,6,5,7615.068855,0,5
7595,f7572,9.524465e+08,9.524465e+08,0.000000,131.84.1.31,24037,45.4.65.127,4942,tcp,-,...,1,40,0,0,-,6,5,7615.069964,0,5
7601,f7578,9.524465e+08,9.524465e+08,0.000000,131.84.1.31,24539,120.81.42.219,4948,tcp,-,...,1,40,0,0,-,6,5,7615.070556,0,5
7602,f7579,9.524465e+08,9.524465e+08,0.000000,131.84.1.31,17031,26.31.94.43,4949,tcp,-,...,1,40,0,0,-,6,5,7615.070635,0,5
7604,f7581,9.524465e+08,9.524465e+08,0.000000,131.84.1.31,20523,101.132.193.240,4951,tcp,-,...,1,40,0,0,-,6,5,7615.070779,0,5
7605,f7582,9.524465e+08,9.524465e+08,0.000000,131.84.1.31,27325,36.227.195.111,4952,tcp,-,...,1,40,0,0,-,6,5,7615.071804,0,5
7606,f7583,9.524465e+08,9.524465e+08,0.000000,131.84.1.31,11320,34.50.49.15,4953,tcp,-,...,1,40,0,0,-,6,5,7615.071880,0,5
7608,f7585,9.524465e+08,9.524465e+08,0.000000,131.84.1.31,4672,16.110.139.90,4955,tcp,-,...,1,40,0,0,-,6,5,7615.072018,0,5
7609,f7586,9.524465e+08,9.524465e+08,0.000000,131.84.1.31,5163,90.5.169.56,4956,tcp,-,...,1,40,0,0,-,6,5,7615.072085,0,5
7610,f7587,9.524465e+08,9.524465e+08,0.000000,131.84.1.31,1602,94.99.146.182,4957,tcp,-,...,1,40,0,0,-,6,5,7615.072159,0,5
